Załadowanie sampli i podział train / validation / test

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, random_split
import h5py
import numpy as np
from tqdm import tqdm
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim


# 1. Automatyczne wykrywanie GPU
device = torch.device("cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu"))
print(f"Używane urządzenie: {device}")

# 2. Definicja zoptymalizowanej klasy Datasetu (całość w RAM)
class FastIsingDataset(Dataset):
    def __init__(self, file_path, assumed_Tc, subsample_fraction=1.0):
        self.assumed_Tc = assumed_Tc
        
        print("Ładowanie całego pliku HDF5 do pamięci RAM... (to potrwa chwilę)")
        with h5py.File(file_path, 'r') as f:
            self.temperatures = f['temperatures'][:]
            all_spins = f['spins'][:] 
            
        self.num_T, self.num_samples, self.L, _ = all_spins.shape
        
        # Opcjonalne zmniejszenie datasetu (subsampling)
        if subsample_fraction < 1.0:
            self.num_samples = int(self.num_samples * subsample_fraction)
            # Ucinamy liczbę próbek dla każdej z temperatur
            all_spins = all_spins[:, :self.num_samples, :, :]
            print(f"Subsampling: Zredukowano do {self.num_samples} próbek na temperaturę.")
            
        self.total_samples = self.num_T * self.num_samples
        
        # Spłaszczamy wymiary [Temperatura, Próbka] na jedną długą listę (Batch, L, L)
        self.spins_flat = all_spins.reshape(self.total_samples, self.L, self.L)
        
        # Etykietowanie: 0 jeśli T <= Tc, 1 jeśli T > Tc
        labels_per_T = (self.temperatures > self.assumed_Tc).astype(int)
        # Rozmnożenie etykiet dla każdej klatki
        self.labels_flat = np.repeat(labels_per_T, self.num_samples)

    def __len__(self):
        return self.total_samples

    def __getitem__(self, idx):
        # Pobieranie błyskawicznie bezpośrednio z RAM-u
        x = self.spins_flat[idx]
        y = self.labels_flat[idx]
        
        # Konwersja na tensor (dodajemy wymiar kanału: 1, wysokość, szerokość)
        x_tensor = torch.tensor(x, dtype=torch.float32).unsqueeze(0)
        y_tensor = torch.tensor(y, dtype=torch.long)
        
        return x_tensor, y_tensor
    
    def close(self):
        # Plik HDF5 zamyka się sam po wyjściu z bloku 'with' w __init__,
        # ale zostawiamy pustą metodę dla kompatybilności ze starym kodem
        pass

# 3. Inicjalizacja datasetu (zaczynamy od 10% na testy architektury)
assumed_Tc = 2.269
dataset = FastIsingDataset("2ising_mc_data_32.h5", assumed_Tc=assumed_Tc, subsample_fraction=1.0)

# 4. Podział na zbiory: Train (70%), Val (15%), Test (15%)
total_len = len(dataset)
train_len = int(0.70 * total_len)
val_len = int(0.15 * total_len)
test_len = total_len - train_len - val_len

# Ustawiamy seed dla reprodukowalności podziału próbek
generator = torch.Generator().manual_seed(42)
train_dataset, val_dataset, test_dataset = random_split(
    dataset, [train_len, val_len, test_len], generator=generator
)

# 5. Tworzenie Loaderów (Większy batch_size = 512 maksymalizuje zużycie GPU)
batch_size = 512
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=0)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=0)

print(f"Zbiór treningowy: {train_len} sampli")
print(f"Zbiór walidacyjny: {val_len} sampli")
print(f"Zbiór testowy: {test_len} sampli")

Model Bazowy

In [ ]:

class IsingCNN_Baseline(nn.Module):
    def __init__(self):
        super(IsingCNN_Baseline, self).__init__()
        
        # Warstwa 1: Wchodzi 1 kanał, wychodzi 8 filtrów
        # kernel_size=3 sprawdza spiny w okienku 3x3
        # padding=1 i mode='circular' zachowują rozmiar 32x32 i periodyczność
        self.conv1 = nn.Conv2d(in_channels=1, out_channels=8, kernel_size=3, 
                               padding=1, padding_mode='circular')
        self.pool1 = nn.MaxPool2d(kernel_size=2, stride=2) # Rozmiar spada do 16x16
        
        # Warstwa 2: Wchodzi 8, wychodzi 16 filtrów
        self.conv2 = nn.Conv2d(in_channels=8, out_channels=16, kernel_size=3, 
                               padding=1, padding_mode='circular')
        self.pool2 = nn.MaxPool2d(kernel_size=2, stride=2) # Rozmiar spada do 8x8
        
        # Warstwy w pełni połączone na końcu (Klasyfikator)
        # Obliczamy rozmiar na wejściu: 16 kanałów * 8 wysokości * 8 szerokości = 1024
        self.fc1 = nn.Linear(16 * 8 * 8, 32)
        # Etykiety to 0 lub 1, więc na końcu potrzebujemy 2 neuronów
        self.fc2 = nn.Linear(32, 2) 

    def forward(self, x):
        # Ekstrakcja cech przestrzennych
        x = self.pool1(F.relu(self.conv1(x)))
        x = self.pool2(F.relu(self.conv2(x)))
        
        # Spłaszczanie tensora przed warstwą liniową
        x = x.view(-1, 16 * 8 * 8)
        
        # Klasyfikacja
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return x

Trening modelu bazowego

In [ ]:

model = IsingCNN_Baseline().to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

epochs = 10
history = {
    'train_loss': [], 'train_acc': [],
    'val_loss': [], 'val_acc': []
}

for epoch in range(1, epochs + 1):
    # --- FAZA TRENINGU ---
    model.train()
    running_loss = 0.0
    correct_train = 0
    total_train = 0
    
    pbar = tqdm(train_loader, desc=f"Epoka {epoch}/{epochs} [Train]")
    for X, y in pbar:
        X, y = X.to(device), y.to(device)
        
        optimizer.zero_grad()
        outputs = model(X)
        loss = criterion(outputs, y)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item() * X.size(0)
        _, predicted = torch.max(outputs.data, 1)
        total_train += y.size(0)
        correct_train += (predicted == y).sum().item()
        
        pbar.set_postfix(loss=loss.item())
        
    epoch_train_loss = running_loss / train_len
    epoch_train_acc = correct_train / total_train
    
    # --- FAZA WALIDACJI ---
    model.eval()
    val_loss = 0.0
    correct_val = 0
    total_val = 0
    
    with torch.no_grad():
        for X, y in val_loader:
            X, y = X.to(device), y.to(device)
            outputs = model(X)
            loss = criterion(outputs, y)
            
            val_loss += loss.item() * X.size(0)
            _, predicted = torch.max(outputs.data, 1)
            total_val += y.size(0)
            correct_val += (predicted == y).sum().item()
            
    epoch_val_loss = val_loss / val_len
    epoch_val_acc = correct_val / total_val
    
    # Zapis do historii wykresów
    history['train_loss'].append(epoch_train_loss)
    history['train_acc'].append(epoch_train_acc)
    history['val_loss'].append(epoch_val_loss)
    history['val_acc'].append(epoch_val_acc)
    
    print(f"\n[Podsumowanie Epoki {epoch}]")
    print(f"Train Loss: {epoch_train_loss:.4f} | Train Acc: {epoch_train_acc:.4f}")
    print(f"Val Loss:   {epoch_val_loss:.4f} | Val Acc:   {epoch_val_acc:.4f}\n")

# ==========================================
# 4. WYKRES
# ==========================================
plt.figure(figsize=(8, 5))
# Oś X to numer epoki, Oś Y to wartość Loss
plt.plot(range(1, epochs + 1), history['train_loss'], label='Train Loss', marker='o', color='blue')
plt.plot(range(1, epochs + 1), history['val_loss'], label='Validation Loss', marker='s', color='orange')

# Upiększanie wykresu
plt.title(f'Model A', fontsize=14)
plt.xlabel('Epoch', fontsize=12)
plt.ylabel('Loss', fontsize=12)
plt.xticks(range(1, epochs + 1)) # Wymusza całkowite numery epok na osi X
plt.legend(fontsize=12)
plt.grid(True, linestyle='--', alpha=0.7)

# Wyświetlenie
plt.tight_layout()
plt.savefig('Model_A.png', dpi = 300)
plt.show()

Model z regularyzacją (BatchNorm, Dropout, Weight decay)

In [ ]:


class IsingCNN_Regularized(nn.Module):
    def __init__(self, dropout_rate=0.5):
        super(IsingCNN_Regularized, self).__init__()
        
        # --- WARSTWA 1 ---
        self.conv1 = nn.Conv2d(1, 8, kernel_size=3, padding=1, padding_mode='circular')
        self.bn1 = nn.BatchNorm2d(8) # Normalizacja 8 wyjściowych kanałów
        self.pool1 = nn.MaxPool2d(2, 2)
        
        # --- WARSTWA 2 ---
        self.conv2 = nn.Conv2d(8, 16, kernel_size=3, padding=1, padding_mode='circular')
        self.bn2 = nn.BatchNorm2d(16) # Normalizacja 16 wyjściowych kanałów
        self.pool2 = nn.MaxPool2d(2, 2)
        
        # --- KLASYFIKATOR (Warstwy gęste) ---
        self.dropout = nn.Dropout(p=dropout_rate) # Regularyzator Dropout
        self.fc1 = nn.Linear(16 * 8 * 8, 32)
        self.fc2 = nn.Linear(32, 2) 

    def forward(self, x):
        # Kolejność: Conv2d -> BatchNorm -> ReLU -> MaxPool
        x = self.pool1(F.relu(self.bn1(self.conv1(x))))
        x = self.pool2(F.relu(self.bn2(self.conv2(x))))
        
        x = x.view(-1, 16 * 8 * 8)
        
        x = self.dropout(x)
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return x



Trening modelu z regularyzacją

In [ ]:

# Parametry regularyzacji do testów
DROPOUT_RATE = 0.5 
WEIGHT_DECAY = 1e-4
LEARNING_RATE = 0.001

model = IsingCNN_Regularized(dropout_rate=DROPOUT_RATE).to(device)
criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)

epochs = 10
history = {
    'train_loss': [], 'train_acc': [],
    'val_loss': [], 'val_acc': []
}

for epoch in range(1, epochs + 1):
    # --- FAZA TRENINGU ---
    model.train()
    running_loss = 0.0
    correct_train = 0
    total_train = 0

    pbar = tqdm(train_loader, desc=f"Epoka {epoch}/{epochs} [Train]")
    for X, y in pbar:
        X, y = X.to(device), y.to(device)
        
        optimizer.zero_grad()
        outputs = model(X)
        loss = criterion(outputs, y)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item() * X.size(0)
        _, predicted = torch.max(outputs.data, 1)
        total_train += y.size(0)
        correct_train += (predicted == y).sum().item()
        
        pbar.set_postfix(loss=loss.item())
        
    epoch_train_loss = running_loss / train_len
    epoch_train_acc = correct_train / total_train
    
    # --- FAZA WALIDACJI ---
    model.eval()
    val_loss = 0.0
    correct_val = 0
    total_val = 0
    
    with torch.no_grad():
        for X, y in val_loader:
            X, y = X.to(device), y.to(device)
            outputs = model(X)
            loss = criterion(outputs, y)
            
            val_loss += loss.item() * X.size(0)
            _, predicted = torch.max(outputs.data, 1)
            total_val += y.size(0)
            correct_val += (predicted == y).sum().item()
            
    epoch_val_loss = val_loss / val_len
    epoch_val_acc = correct_val / total_val
    
    # Zapis do historii wykresów
    history['train_loss'].append(epoch_train_loss)
    history['train_acc'].append(epoch_train_acc)
    history['val_loss'].append(epoch_val_loss)
    history['val_acc'].append(epoch_val_acc)
    
    print(f"\n[Podsumowanie Epoki {epoch}]")
    print(f"Train Loss: {epoch_train_loss:.4f} | Train Acc: {epoch_train_acc:.4f}")
    print(f"Val Loss:   {epoch_val_loss:.4f} | Val Acc:   {epoch_val_acc:.4f}\n")

# ==========================================
# 4. RYSOWANIE WYKRESU 
# ==========================================
plt.figure(figsize=(8, 5))
# Oś X to numer epoki, Oś Y to wartość Loss
plt.plot(range(1, epochs + 1), history['train_loss'], label='Train Loss', marker='o', color='blue')
plt.plot(range(1, epochs + 1), history['val_loss'], label='Validation Loss', marker='s', color='orange')

# Upiększanie wykresu
plt.title(f'Model B', fontsize=14)
plt.xlabel('Epoch', fontsize=12)
plt.ylabel('Loss', fontsize=12)
plt.xticks(range(1, epochs + 1)) # Wymusza całkowite numery epok na osi X
plt.legend(fontsize=12)
plt.grid(True, linestyle='--', alpha=0.7)

# Wyświetlenie
plt.tight_layout()
plt.savefig('savepls_modelB_loss.png', dpi = 300)
plt.show()

Model C - Głębszy + regularyzacja

In [ ]:
import torch.nn as nn
import torch.nn.functional as F

class IsingCNN_DeepNarrow(nn.Module):
    def __init__(self, dropout_rate=0.5):
        super(IsingCNN_DeepNarrow, self).__init__()
        
        # --- WARSTWA 1 (Wejście: 32x32) ---
        # Tylko 8 filtrów!
        self.conv1 = nn.Conv2d(1, 8, kernel_size=3, padding=1, padding_mode='circular')
        self.bn1 = nn.BatchNorm2d(8)
        self.pool1 = nn.MaxPool2d(2, 2) # Wyjście: 16x16
        
        # --- WARSTWA 2 (Wejście: 16x16) ---
        self.conv2 = nn.Conv2d(8, 16, kernel_size=3, padding=1, padding_mode='circular')
        self.bn2 = nn.BatchNorm2d(16)
        self.pool2 = nn.MaxPool2d(2, 2) # Wyjście: 8x8
        
        # --- WARSTWA 3 (Wejście: 8x8) ---
        # Zostajemy przy 16 filtrach, pogłębiamy analizę bez zwiększania szerokości
        self.conv3 = nn.Conv2d(16, 16, kernel_size=3, padding=1, padding_mode='circular')
        self.bn3 = nn.BatchNorm2d(16)
        self.pool3 = nn.MaxPool2d(2, 2) # Wyjście: 4x4
        
        # --- WARSTWA 4 (Wejście: 4x4) ---
        # Ostatni krok przed klasyfikatorem
        self.conv4 = nn.Conv2d(16, 32, kernel_size=3, padding=1, padding_mode='circular')
        self.bn4 = nn.BatchNorm2d(32)
        self.pool4 = nn.MaxPool2d(2, 2) # Wyjście: 2x2
        
        # --- KLASYFIKATOR (Wąskie gardło) ---
        self.dropout = nn.Dropout(p=dropout_rate)
        
        # Spłaszczamy: 32 kanały * 2 wysokość * 2 szerokość = zaledwie 128 parametrów wejściowych!
        # (W starym Modelu C było ich tutaj 1024)
        self.fc1 = nn.Linear(128, 32) # Redukcja z 128 do zaledwie 32 neuronów
        self.bn_fc = nn.BatchNorm1d(32)
        self.fc2 = nn.Linear(32, 2)

    def forward(self, x):
        # 4 głębokie kroki
        x = self.pool1(F.relu(self.bn1(self.conv1(x))))
        x = self.pool2(F.relu(self.bn2(self.conv2(x))))
        x = self.pool3(F.relu(self.bn3(self.conv3(x))))
        x = self.pool4(F.relu(self.bn4(self.conv4(x))))
        
        # Klasyfikacja
        x = x.view(-1, 32 * 2 * 2) 
        x = self.dropout(x)
        x = F.relu(self.bn_fc(self.fc1(x)))
        x = self.fc2(x)
        return x

Trening modelu C

In [ ]:
# ==========================================
# 2. INICJALIZACJA I HIPERPARAMETRY
# ==========================================
DROPOUT_RATE = 0.5
WEIGHT_DECAY = 1e-4 
LEARNING_RATE = 0.001
EPOCHS = 10          

model = IsingCNN_DeepNarrow(dropout_rate=DROPOUT_RATE).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)

history = {
    'train_loss': [], 'train_acc': [],
    'val_loss': [], 'val_acc': []
}

print(f"Rozpoczynam trening Modelu C (Głęboki z Regularyzacją) na urządzeniu: {device}...")

# ==========================================
# 3. PĘTLA TRENINGOWA
# ==========================================
for epoch in range(1, EPOCHS + 1):
    # --- FAZA TRENINGU ---
    model.train()
    running_loss = 0.0
    correct_train = 0
    total_train = 0
    
    # tqdm pokaże ładny pasek postępu wewnątrz epoki
    pbar = tqdm(train_loader, desc=f"Epoka {epoch}/{EPOCHS} [Train]")
    for X, y in pbar:
        X, y = X.to(device), y.to(device)
        
        optimizer.zero_grad()
        outputs = model(X)
        loss = criterion(outputs, y)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item() * X.size(0)
        _, predicted = torch.max(outputs.data, 1)
        total_train += y.size(0)
        correct_train += (predicted == y).sum().item()
        
        pbar.set_postfix(loss=loss.item())
        
    epoch_train_loss = running_loss / train_len
    epoch_train_acc = correct_train / total_train
    
    # --- FAZA WALIDACJI ---
    model.eval()
    val_loss = 0.0
    correct_val = 0
    total_val = 0
    
    with torch.no_grad():
        for X, y in val_loader:
            X, y = X.to(device), y.to(device)
            outputs = model(X)
            loss = criterion(outputs, y)
            
            val_loss += loss.item() * X.size(0)
            _, predicted = torch.max(outputs.data, 1)
            total_val += y.size(0)
            correct_val += (predicted == y).sum().item()
            
    epoch_val_loss = val_loss / val_len
    epoch_val_acc = correct_val / total_val
    
    # Zapis do historii wykresów
    history['train_loss'].append(epoch_train_loss)
    history['train_acc'].append(epoch_train_acc)
    history['val_loss'].append(epoch_val_loss)
    history['val_acc'].append(epoch_val_acc)
    
    print(f"\n[Podsumowanie Epoki {epoch}]")
    print(f"Train Loss: {epoch_train_loss:.4f} | Train Acc: {epoch_train_acc:.4f}")
    print(f"Val Loss:   {epoch_val_loss:.4f} | Val Acc:   {epoch_val_acc:.4f}\n")

# ==========================================
# 4. RYSOWANIE WYKRESU 
# ==========================================
plt.figure(figsize=(8, 5))
# Oś X to numer epoki, Oś Y to wartość Loss
plt.plot(range(1, EPOCHS + 1), history['train_loss'], label='Train Loss', marker='o', color='blue')
plt.plot(range(1, EPOCHS + 1), history['val_loss'], label='Validation Loss', marker='s', color='orange')

# Upiększanie wykresu
plt.title('Model C', fontsize=14)
plt.xlabel('Epoch', fontsize=12)
plt.ylabel('Loss', fontsize=12)
plt.xticks(range(1, EPOCHS + 1)) # Wymusza całkowite numery epok na osi X
plt.legend(fontsize=12)
plt.grid(True, linestyle='--', alpha=0.7)

# Wyświetlenie i zapis
plt.tight_layout()
plt.savefig('modelC_loss.png', dpi=300)
plt.show()

Learning By Confusion

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt


def update_Tc_patch(self, new_Tc):
    self.assumed_Tc = new_Tc
    labels_per_T = (self.temperatures > self.assumed_Tc).astype(int)
    self.labels_flat = np.repeat(labels_per_T, self.num_samples)

FastIsingDataset.update_Tc = update_Tc_patch

# ==========================================
# 2. GŁÓWNA PĘTLA: LEARNING BY CONFUSION (POPRAWIONA)
# ==========================================

critical_temperatures = np.linspace(1.0, 3.5, 25)
accuracies = []


EPOCHS = 10 

print("\n" + "="*50)
print(f"START LEARNING BY CONFUSION na {len(critical_temperatures)} temperaturach")
print("="*50)

for tc in critical_temperatures:
    print(f"\n---> Testowanie założonego Tc = {tc:.3f}")
    
    dataset.update_Tc(tc)
    
    model = IsingCNN_Regularized(dropout_rate=0.5).to(device)
    criterion = nn.CrossEntropyLoss()
    
    # ZMNIEJSZONY LEARNING RATE: 0.0003 zamiast 0.001
    optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-4)
    
    model.train()
    for epoch in range(1, EPOCHS + 1):
        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            optimizer.zero_grad()
            outputs = model(X_batch)
            loss = criterion(outputs, y_batch)
            loss.backward()
            optimizer.step()
            
    # Ewaluacja
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for X_batch, y_batch in test_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            outputs = model(X_batch)
            _, predicted = torch.max(outputs.data, 1)
            total += y_batch.size(0)
            correct += (predicted == y_batch).sum().item()
            
    acc = correct / total
    accuracies.append(acc)
    print(f"Zakończono. Accuracy na zbiorze testowym: {acc:.4f}")

# ==========================================
# 3. RYSOWANIE FINAŁOWEGO WYKRESU
# ==========================================
plt.figure(figsize=(10, 6))
plt.plot(critical_temperatures, accuracies, marker='o', linestyle='-', color='purple', linewidth=2, markersize=8)

theoretical_Tc = 2.269
plt.axvline(x=theoretical_Tc, color='red', linestyle='--', label=f' $T_c \\approx {theoretical_Tc}$')

plt.title('Learning by Confusion', fontsize=16)
plt.xlabel("$T_c$", fontsize=14)
plt.ylabel('Test accuracy', fontsize=14)
plt.grid(True, linestyle='--', alpha=0.7)
plt.legend(fontsize=12)
plt.tight_layout()

# Zapis wykresu i danych surowych do raportu
plt.savefig('learning_by_confusion_w_shape.png', dpi=300)
plt.show()

np.save('Blbc_temperatures.npy', critical_temperatures)
np.save('Blbc_accuracies.npy', accuracies)
print("\nGotowe! Wyniki i wykres zostały zapisane.")

Model C - dla siatki 16x16

In [ ]:
import torch.nn as nn
import torch.nn.functional as F

class IsingCNN_DeepNarrow_16x16(nn.Module):
    def __init__(self, dropout_rate=0.5):
        super(IsingCNN_DeepNarrow_16x16, self).__init__()
        
        # Warstwy konwolucyjne zostają BEZ ZMIAN
        self.conv1 = nn.Conv2d(1, 8, kernel_size=3, padding=1, padding_mode='circular')
        self.bn1 = nn.BatchNorm2d(8)
        self.pool1 = nn.MaxPool2d(2, 2) 
        
        self.conv2 = nn.Conv2d(8, 16, kernel_size=3, padding=1, padding_mode='circular')
        self.bn2 = nn.BatchNorm2d(16)
        self.pool2 = nn.MaxPool2d(2, 2) 
        
        self.conv3 = nn.Conv2d(16, 16, kernel_size=3, padding=1, padding_mode='circular')
        self.bn3 = nn.BatchNorm2d(16)
        self.pool3 = nn.MaxPool2d(2, 2) 
        
        self.conv4 = nn.Conv2d(16, 32, kernel_size=3, padding=1, padding_mode='circular')
        self.bn4 = nn.BatchNorm2d(32)
        self.pool4 = nn.MaxPool2d(2, 2) 
        
        self.dropout = nn.Dropout(p=dropout_rate)
        
        # --- ZMIANA TUTAJ ---
        # Po 4 warstwach poolingu z 16x16 zostaje 1x1. 
        # Mamy 32 kanały, więc 32 * 1 * 1 = 32 wejścia (a nie 128)
        self.fc1 = nn.Linear(32, 32) 
        self.bn_fc = nn.BatchNorm1d(32)
        self.fc2 = nn.Linear(32, 2)

    def forward(self, x):
        x = self.pool1(F.relu(self.bn1(self.conv1(x))))
        x = self.pool2(F.relu(self.bn2(self.conv2(x))))
        x = self.pool3(F.relu(self.bn3(self.conv3(x))))
        x = self.pool4(F.relu(self.bn4(self.conv4(x))))
        
        # --- ZMIANA TUTAJ ---
        # Spłaszczamy tensor do (batch_size, 32)
        x = x.view(-1, 32 * 1 * 1) 
        x = self.dropout(x)
        x = F.relu(self.bn_fc(self.fc1(x)))
        x = self.fc2(x)
        return x

In [ ]:
# ==========================================
# 2. INICJALIZACJA I HIPERPARAMETRY
# ==========================================
DROPOUT_RATE = 0.5
WEIGHT_DECAY = 1e-4  
LEARNING_RATE = 0.001
EPOCHS = 10       

model = IsingCNN_DeepNarrow_16x16(dropout_rate=DROPOUT_RATE).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)

history = {
    'train_loss': [], 'train_acc': [],
    'val_loss': [], 'val_acc': []
}

print(f"Rozpoczynam trening Modelu C (Głęboki z Regularyzacją) na urządzeniu: {device}...")

# ==========================================
# 3. PĘTLA TRENINGOWA
# ==========================================
for epoch in range(1, EPOCHS + 1):
    # --- FAZA TRENINGU ---
    model.train()
    running_loss = 0.0
    correct_train = 0
    total_train = 0
    
    # tqdm pokaże ładny pasek postępu wewnątrz epoki
    pbar = tqdm(train_loader, desc=f"Epoka {epoch}/{EPOCHS} [Train]")
    for X, y in pbar:
        X, y = X.to(device), y.to(device)
        
        optimizer.zero_grad()
        outputs = model(X)
        loss = criterion(outputs, y)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item() * X.size(0)
        _, predicted = torch.max(outputs.data, 1)
        total_train += y.size(0)
        correct_train += (predicted == y).sum().item()
        
        pbar.set_postfix(loss=loss.item())
        
    epoch_train_loss = running_loss / train_len
    epoch_train_acc = correct_train / total_train
    
    # --- FAZA WALIDACJI ---
    model.eval()
    val_loss = 0.0
    correct_val = 0
    total_val = 0
    
    with torch.no_grad():
        for X, y in val_loader:
            X, y = X.to(device), y.to(device)
            outputs = model(X)
            loss = criterion(outputs, y)
            
            val_loss += loss.item() * X.size(0)
            _, predicted = torch.max(outputs.data, 1)
            total_val += y.size(0)
            correct_val += (predicted == y).sum().item()
            
    epoch_val_loss = val_loss / val_len
    epoch_val_acc = correct_val / total_val
    
    # Zapis do historii wykresów
    history['train_loss'].append(epoch_train_loss)
    history['train_acc'].append(epoch_train_acc)
    history['val_loss'].append(epoch_val_loss)
    history['val_acc'].append(epoch_val_acc)
    
    print(f"\n[Podsumowanie Epoki {epoch}]")
    print(f"Train Loss: {epoch_train_loss:.4f} | Train Acc: {epoch_train_acc:.4f}")
    print(f"Val Loss:   {epoch_val_loss:.4f} | Val Acc:   {epoch_val_acc:.4f}\n")

# ==========================================
# 4. RYSOWANIE WYKRESU (TRAIN I VAL NA JEDNYM)
# ==========================================
plt.figure(figsize=(8, 5))
# Oś X to numer epoki, Oś Y to wartość Loss
plt.plot(range(1, EPOCHS + 1), history['train_loss'], label='Train Loss', marker='o', color='blue')
plt.plot(range(1, EPOCHS + 1), history['val_loss'], label='Validation Loss', marker='s', color='orange')

# Upiększanie wykresu
plt.title('Model C', fontsize=14)
plt.xlabel('Epoch', fontsize=12)
plt.ylabel('Loss', fontsize=12)
plt.xticks(range(1, EPOCHS + 1)) # Wymusza całkowite numery epok na osi X
plt.legend(fontsize=12)
plt.grid(True, linestyle='--', alpha=0.7)

# Wyświetlenie i zapis
plt.tight_layout()
plt.savefig('modelC_loss.png', dpi=300)
plt.show()

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt


def update_Tc_patch(self, new_Tc):
    self.assumed_Tc = new_Tc
    labels_per_T = (self.temperatures > self.assumed_Tc).astype(int)
    self.labels_flat = np.repeat(labels_per_T, self.num_samples)

FastIsingDataset.update_Tc = update_Tc_patch

# ==========================================
# 2. GŁÓWNA PĘTLA: LEARNING BY CONFUSION
# ==========================================

critical_temperatures = np.linspace(1.0, 3.5, 25)
accuracies = []


EPOCHS = 10 

print("\n" + "="*50)
print(f"START LEARNING BY CONFUSION na {len(critical_temperatures)} temperaturach")
print("="*50)

for tc in critical_temperatures:
    print(f"\n---> Testowanie założonego Tc = {tc:.3f}")
    
    dataset.update_Tc(tc)
    
    model = IsingCNN_DeepNarrow_16x16(dropout_rate=0.5).to(device)
    criterion = nn.CrossEntropyLoss()
    
    # ZMNIEJSZONY LEARNING RATE: 0.0003 zamiast 0.001
    optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-4)
    
    model.train()
    for epoch in range(1, EPOCHS + 1):
        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            optimizer.zero_grad()
            outputs = model(X_batch)
            loss = criterion(outputs, y_batch)
            loss.backward()
            optimizer.step()
            
    # Ewaluacja
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for X_batch, y_batch in test_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            outputs = model(X_batch)
            _, predicted = torch.max(outputs.data, 1)
            total += y_batch.size(0)
            correct += (predicted == y_batch).sum().item()
            
    acc = correct / total
    accuracies.append(acc)
    print(f"Zakończono. Accuracy na zbiorze testowym: {acc:.4f}")

# ==========================================
# 3. RYSOWANIE FINAŁOWEGO WYKRESU
# ==========================================
plt.figure(figsize=(10, 6))
plt.plot(critical_temperatures, accuracies, marker='o', linestyle='-', color='purple', linewidth=2, markersize=8)

# Linia pionowa oznaczająca teoretyczne Tc dla modelu Isinga 2D
theoretical_Tc = 2.269
plt.axvline(x=theoretical_Tc, color='red', linestyle='--', label=f' $T_c \\approx {theoretical_Tc}$')

plt.title('Learning by Confusion', fontsize=16)
plt.xlabel("$T_c$", fontsize=14)
plt.ylabel('Test accuracy', fontsize=14)
plt.grid(True, linestyle='--', alpha=0.7)
plt.legend(fontsize=12)
plt.tight_layout()

# Zapis wykresu i danych surowych do raportu
plt.savefig('1616learning_by_confusion_w_shape.png', dpi=300)
plt.show()

np.save('16lbc_temperatures.npy', critical_temperatures)
np.save('16lbc_accuracies.npy', accuracies)
print("\nGotowe! Wyniki i wykres zostały zapisane.")